# Unsupervised Learning

In this notebook we will apply:
- PCA
- KMeans
- DBSCAN
- Agglomerative Clustering

I am keeping the code short and simple.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import silhouette_score

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
df = pd.read_csv('Dataset/archive(2)/train.csv').sample(5000, random_state=42)
print(df.shape)
df.head()

In [ ]:
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
df = df.drop(columns=['employee_id', 'is_promoted'])
num = df.select_dtypes(include=['int64', 'float64']).columns
cat = df.select_dtypes(include='object').columns

pre = ColumnTransformer([
    ('n', Pipeline([('fill', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num),
    ('c', Pipeline([('fill', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat)
])

x = pre.fit_transform(df)
x.shape

## PCA

PCA helps us reduce the data into 2 columns so we can plot it easily.

In [ ]:
pca = PCA(n_components=2, random_state=42)
p2 = pca.fit_transform(x)
p = pd.DataFrame(p2, columns=['PC1', 'PC2'])

print('Explained variance:', pca.explained_variance_ratio_)
sns.scatterplot(data=p, x='PC1', y='PC2', s=35)
plt.title('PCA Plot')
plt.show()

## KMeans

First we check a few values of `k`, then we choose the best one.

In [ ]:
ks = range(2, 7)
sil = []
inn = []

for k in ks:
    m = KMeans(n_clusters=k, random_state=42, n_init=10)
    y = m.fit_predict(x)
    sil.append(silhouette_score(x, y))
    inn.append(m.inertia_)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(ks), inn, marker='o')
ax[0].set_title('Elbow Plot')
ax[1].plot(list(ks), sil, marker='o', color='orange')
ax[1].set_title('Silhouette Score')
plt.show()

k = list(ks)[np.argmax(sil)]
print('Best k =', k)

In [ ]:
km = KMeans(n_clusters=k, random_state=42, n_init=10)
p['kmeans'] = km.fit_predict(x)
print('KMeans silhouette on PCA:', round(silhouette_score(p2, p['kmeans']), 3))

sns.scatterplot(data=p, x='PC1', y='PC2', hue='kmeans', palette='Set1', s=40)
plt.title('KMeans Clusters')
plt.show()

print(p['kmeans'].value_counts().sort_index())

## Hierarchical View

A dendrogram helps us see how clusters are joining together. I am using a smaller PCA sample here so the plot stays readable.

In [ ]:
tree = linkage(p2[:300], method='ward')

plt.figure(figsize=(10, 5))
dendrogram(tree, no_labels=True, color_threshold=None)
plt.title('Dendrogram View')
plt.xlabel('Sample points')
plt.ylabel('Distance')
plt.show()

## DBSCAN

DBSCAN groups close points together and can mark some points as noise.

In [ ]:
db = DBSCAN(eps=0.2, min_samples=8)
p['dbscan'] = db.fit_predict(p2)

sns.scatterplot(data=p, x='PC1', y='PC2', hue='dbscan', palette='tab10', s=40)
plt.title('DBSCAN Clusters')
plt.show()

print(p['dbscan'].value_counts().sort_index())

## Agglomerative Clustering

This method joins similar points step by step.

In [ ]:
ag = AgglomerativeClustering(n_clusters=k)
p['agg'] = ag.fit_predict(x)
print('Agglomerative silhouette on PCA:', round(silhouette_score(p2, p['agg']), 3))

sns.scatterplot(data=p, x='PC1', y='PC2', hue='agg', palette='Dark2', s=40)
plt.title('Agglomerative Clusters')
plt.show()

print(p['agg'].value_counts().sort_index())

In [ ]:
pd.DataFrame({
    'model': ['KMeans', 'DBSCAN', 'Agglomerative'],
    'groups': [p['kmeans'].nunique(), p['dbscan'].nunique(), p['agg'].nunique()]
})